In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

HOSP = Path("../data/mimiciv-3.1/hosp")
patients = pd.read_csv(HOSP / "patients.csv", parse_dates=["dod"])
admissions = pd.read_csv(HOSP / "admissions.csv", parse_dates=["admittime", "dischtime", "deathtime", "edregtime", "edouttime"])

In [ ]:
# merged admissions and patients
admissions_65plus = pd.merge(admissions, patients, on="subject_id", how="left", validate="many_to_one") 
# calc the age for each admission
admissions_65plus["age_at_admission"] = admissions_65plus["anchor_age"] + admissions_65plus["admittime"].dt.year - admissions_65plus["anchor_year"]
# filtered the merged df by patients that are at leas 65 years old and didn't pass away in the hospital
admissions_65plus = admissions_65plus[(admissions_65plus["age_at_admission"] >= 65) & (admissions_65plus["hospital_expire_flag"] == 0)].copy()

# filtering the columns
admissions_65plus = admissions_65plus[["subject_id", "hadm_id", "admittime", "dischtime", "admission_type", "discharge_location", "race",
                                       "edregtime", "edouttime",  "gender", "anchor_age", "anchor_year", "dod", "age_at_admission"]]
admissions_65plus = admissions_65plus.reset_index(drop=True)
display(admissions_65plus.shape, admissions_65plus.head(), admissions_65plus.info())

# plotting numbers of stays by age
max_age = admissions_65plus["age_at_admission"].max()
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(admissions_65plus["age_at_admission"], bins=range(65, max_age + 2), edgecolor="white", color="steelblue")
ax.set_xlabel("Age at admission (years)")
ax.set_ylabel("Number of hospital stays")
ax.set_title("Age at admission, stays of patients aged 65+")
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
#all admission types
display(admissions_65plus["admission_type"].value_counts())

#admissions where discharge locations are NaN
display(admissions_65plus[admissions_65plus["discharge_location"].isna()]["admission_type"].value_counts())

admissions_65plus["admission_hours"] = (admissions_65plus["dischtime"] - admissions_65plus["admittime"]).dt.total_seconds() / 3600
admissions_65plus.groupby("admission_type")["admission_hours"].median().sort_values()

In [ ]:
OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)
admissions_65plus.to_parquet(OUT / "admissions_65plus.parquet", index=False)